In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm

In [ ]:
import torch
from transformers import AutoTokenizer, EsmModel
from torch.utils.data import DataLoader
list_num_model=['t6_8M', 't12_35M', 't30_150M', 't33_650M', 't36_3B']


num_model = list_num_model[3]

MODEL_NAME = f"facebook/esm2_{num_model}_UR50D"

print(f"Загрузка модели {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = EsmModel.from_pretrained(MODEL_NAME)
#model = EsmModel.from_pretrained(MODEL_NAME, torch_dtype=torch.float16)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

/home/admingwi/anaconda3/envs/torch_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Загрузка модели facebook/esm2_t6_8M_UR50D...


Loading weights: 100%|██████████| 107/107 [00:00<00:00, 598.95it/s, Materializing param=encoder.layer.5.output.dense.weight]                      
EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
pooler.dense.weight         | MISSING    | 
pooler.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


EsmModel(
  (embeddings): EsmEmbeddings(
    (word_embeddings): Embedding(33, 320, padding_idx=1)
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): EsmEncoder(
    (layer): ModuleList(
      (0-5): 6 x EsmLayer(
        (attention): EsmAttention(
          (self): EsmSelfAttention(
            (query): Linear(in_features=320, out_features=320, bias=True)
            (key): Linear(in_features=320, out_features=320, bias=True)
            (value): Linear(in_features=320, out_features=320, bias=True)
            (rotary_embeddings): RotaryEmbedding()
          )
          (output): EsmSelfOutput(
            (dense): Linear(in_features=320, out_features=320, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
          (LayerNorm): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
        )
        (intermediate): EsmIntermediate(
          (dense): Linear(in_features=320, out_features=1280, bias=True)
        )
        (output): EsmOutput(
        

In [ ]:
import re
import numpy as np
import pandas as pd

test_seq = "AC-HIK-LMN_"

def clean_peptide(sequence):
    """
  Cleans the peptide sequence:
  - Removes ONLY the specified N- and C-terminal modifications
  - Removes parentheses and their contents
  - Removes all hyphens and spaces
  - Ensures that only uppercase canonical amino acids remain
  - Returns NaN for any mismatches
    """
    if not isinstance(sequence, str) or pd.isna(sequence):
        return np.nan

    seq = sequence.strip()

    # 1. Remove ONLY specific N-terminal modifications
    n_terminal_modifications = ['CH3COO-', 'Ac-']

    # We check each modification (case-sensitive)
    for mod in n_terminal_modifications:
        if seq.startswith(mod):
            seq = seq[len(mod):]
            break

    # 2. Remove ONLY specific C-terminal modifications
    c_terminal_modifications = ['-NH2', '-NH₂', '-OH', '-COOH', '-CONH2']

    # check each modification (case-sensitive)
    for mod in c_terminal_modifications:
        if seq.endswith(mod):
            seq = seq[:-len(mod)]
            break

    # 3. Remove the brackets and their contents
    seq = re.sub(r'[\(\[{][^\)\]}]*[\)\]}]', '', seq)

    # 4. Remove all hyphens and spaces
    seq = seq.replace('-', '').replace(' ', '')

    # 5. Checks
    if not seq:  # Empty str
        return np.nan

    if not seq.isupper():  # There are lowercase letters
        return np.nan

    # 6. Testing for canonical amino acids
    canonical_amino_acids = set('ACDEFGHIKLMNPQRSTVWY')
    for aa in seq:
        if aa not in canonical_amino_acids:
            return np.nan

    return seq
    return seq_upper

print("Basic function:")
print(f"Input: '{test_seq}' -> Output: '{clean_peptide(test_seq)}'")
print()

In [ ]:
path = 'Data'
folder = 'Toxicity'
data_name = 'NTX'
set_name = 'train'
func_y = 'Tox'

In [ ]:
path = 'Data'
folder = 'Pampa'
data_name = 'pampa_b'
set_name = 'train'
func_y = 'PAMPA'
emb_shape = 1280

In [ ]:
path = 'Data'
folder = 'Half_life'
data_name = 'Half_life'
set_name = 'train_organ'
#func_y = 'Half-life, h (Homo sapiens)'
func_y = 'Half-life, h'
emb_shape = 1280

In [ ]:
path = 'Data'
folder = 'Cell_p'
data_name = 'Cell_p'
set_name = 'test'
func_y = 'CellP'

In [ ]:
path = 'Data'
folder = 'Caco'
data_name = 'caco_b'
set_name = 'train'
func_y = 'Caco-2_prm'

In [ ]:
hl_df = pd.read_excel(f'{path}/{folder}/{data_name}_{set_name}.xlsx')
hl_df

,sources,Name,Canonical_smiles,PAMPA,Cyclic/Linear (checked),Functional_activity,PK_groups,Sequence
0,CycPeptMPDB,hexa_1045,CC(C)C[C@@H]1NC(=O)[C@@H](C)N(C)C(=O)[C@H](Cc2...,1,Cyclic,NaN,NaN,NaN
1,CycPeptMPDB,hexa_723,CC(C)C[C@@H]1NC(=O)[C@@H]2CCCN2C(=O)[C@H](CC(C...,0,Cyclic,NaN,NaN,NaN
2,CycPeptMPDB,LB05,CCC[C@@H]1NC(=O)CN(CC)C(=O)[C@H](CC(C)C)NC(=O)...,0,Cyclic,NaN,NaN,NaN
3,CycPeptMPDB,hexa_798,CC(C)C[C@H]1C(=O)N[C@@H](Cc2ccccc2)CC(=O)N2CCC...,1,Cyclic,NaN,NaN,NaN
4,CycPeptMPDB,hepta_836,CCCN1CC(=O)N[C@H](CC(C)C)C(=O)N(Cc2ccccc2)CC(=...,0,Cyclic,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
5539,CycPeptMPDB,hexa_975,CC(C)C[C@H]1C(=O)N[C@@H](Cc2ccccc2)CC(=O)N2CCC...,0,Cyclic,NaN,NaN,NaN
5540,CycPeptMPDB,L1_4.3.2.3.3.1,CC(=O)N1CCC[C@@H]1C(=O)N(C)[C@@H](CC(C)C)C(=O)...,0,Cyclic,NaN,NaN,NaN
5541,CycPeptMPDB,L1_9.2.4.3.3.2,CC(=O)N1CCC[C@H]1C(=O)N(C)[C@@H](CC(C)C)C(=O)N...,0,Cyclic,NaN,NaN,NaN
5542,CycPeptMPDB,hepta_149,CC(C)C[C@H]1C(=O)N[C@@H](Cc2ccccc2)C(=O)N2CCC[...,0,Cyclic,NaN,NaN,NaN


In [ ]:
df = pd.DataFrame(np.zeros((hl_df.shape[0], 1280)))
df['Sequence']=hl_df.Sequence
df['y']=hl_df[func_y]
num_model = 't33_650M'
df

,0,1,2,3,4,5,6,7,8,9,...,1272,1273,1274,1275,1276,1277,1278,1279,Sequence,y
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,1
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,1
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5539,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0
5540,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0
5541,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0
5542,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0


In [ ]:
df.to_csv(f'{path}/{folder}/{data_name}_{set_name}_esm2_{num_model}.csv', index=None)

In [ ]:
categories = ['Homo sapiens', 'Rattus norvegicus', 'Mus musculus', 'Macaca fascicularis', 'Canis lupus']  # Порядок важен!
mapping = {cat: i for i, cat in enumerate(categories)}
hl_df['Org_ind'] = hl_df['Organism'].map(mapping)

In [ ]:
#hl_df['Org_ind']=0.0

In [ ]:
def get_protein_embeddings(sequences):
    """
    Converts a list of peptide sequences into a matrix of features (embeddings).
    """
    model.eval()
    embeddings = []

    # Processing sequences
    with torch.no_grad():
        for seq in sequences:
            clean_seq = clean_peptide(seq)
            if pd.isna(clean_seq):
                embeddings.append(np.zeros(emb_shape))
                continue
            inputs = tokenizer(clean_seq, return_tensors="pt", padding=True, truncation=True).to(device)

            outputs = model(**inputs)

            # outputs.last_hidden_state has dimension (1, Seq_Len, Hidden_Dim)
            # take the average for all amino acids, excluding special symbols (beginning and end)
            token_embeddings = outputs.last_hidden_state

            attention_mask = inputs['attention_mask']

            # multiply by the mask to zero out the padding and calculate the average
            input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
            sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
            sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)

            mean_pooling = sum_embeddings / sum_mask

            # Convert to a numpy array and add to the list
            embeddings.append(mean_pooling.cpu().numpy()[0])

    return np.array(embeddings)

# --- Example of use ---
peptides = [
    "ACDEF",           # Short peptide
    "MVLSPADKTNVKAA",  # Long peptide
    "GGGG"             # Polyglycine
]

print("Generating embeddings...")
features = np.array(get_protein_embeddings(hl_df.Sequence))

print(f"\nDimension of the feature matrix: {features.shape}")


x_df = pd.DataFrame(features)
x_df['Sequence'] = hl_df['Sequence']
#spec_target = np.log10(hl_df[func_y])
#x_df['y'] = spec_target
#x_df['Org_ind'] = hl_df['Org_ind']
x_df['y'] = hl_df['Org_ind']
x_df.to_csv(f'{path}/{folder}/{data_name}_{set_name}_esm2_{num_model}.csv', index=None)

In [ ]:
x_df

,0,1,2,3,4,5,6,7,8,9,...,1273,1274,1275,1276,1277,1278,1279,Sequence,y,Org_ind
0,0.005252,0.048175,-0.171349,0.081066,-0.061782,-0.066760,0.132874,0.001083,0.041856,0.065142,...,0.189804,-0.056302,-0.241251,0.021757,-0.235557,-0.076366,-0.087694,SYSMEHFRWGKPVGKKRRPVKVYPNGAEDESAEAFPLEF,-0.602060,0.0
1,0.044234,0.025169,0.105580,0.050535,-0.082038,-0.015482,-0.018064,0.180011,0.079135,0.105745,...,0.122728,-0.006791,0.134502,0.014334,0.001887,-0.055464,-0.033896,KCNTATCATQRLANFLVHSSNNFGPILPPTNVGSNTY-NH2,-0.096910,0.0
2,-0.003458,-0.111105,0.046987,0.107577,-0.029715,-0.130306,0.063758,0.065933,0.207552,0.206298,...,-0.009714,-0.048501,-0.068943,0.088981,-0.098612,0.104779,0.136849,MTPLGPASSLPQSFLLKCLEQVRKIQGDGAALQEKLCATYKLCHPE...,1.672098,0.0
3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,Hyp-dPhg-wK-Y(Bn)-F,1.079181,0.0
4,-0.024303,-0.000333,-0.023519,0.110215,-0.036727,-0.122598,-0.018346,0.088025,0.036919,0.048873,...,-0.037462,0.009484,0.061497,0.091440,-0.035051,-0.026386,-0.000153,MSYNLLGFLQRSSNFQCQKLLWQLNGRLEYCLKDRMNFDIPEEIKQ...,1.892095,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
286,0.020721,0.013345,-0.067855,0.072899,-0.087602,0.004342,-0.076830,0.152373,0.065472,-0.059355,...,0.066286,0.011481,0.153322,0.094468,-0.090317,-0.119199,0.005517,YAEGTFISDYSIAMDKIHQQDFVNWLLAQKGKKNDWKHNITQ,0.778151,0.0
287,0.054229,0.022918,0.031372,-0.020893,-0.041319,-0.045668,-0.161293,0.073997,-0.083483,0.042044,...,0.207303,0.056925,0.019525,0.060205,0.130185,0.016716,-0.005617,YGRKKRRQRRR,0.824126,0.0
288,0.061354,0.075946,0.065789,0.122620,0.005872,-0.087753,-0.134428,0.272418,0.095205,-0.045426,...,0.093366,0.013334,0.148254,-0.049277,-0.128064,0.061513,-0.092456,YPFP-NH2,-0.356547,0.0
289,0.038600,0.060176,0.088626,0.083822,-0.103019,-0.048487,-0.099295,0.213962,0.119494,-0.070947,...,0.113856,0.022454,0.140820,-0.035372,0.047142,0.018137,-0.023592,YVMGHFRWDRFG-NH2,-1.522879,0.0


In [ ]:


# List of sequences
sequences = hl_df['Sequence'].tolist()

BATCH_SIZE = 16
dataloader = DataLoader(sequences, batch_size=BATCH_SIZE, shuffle=False)

x = []

with torch.no_grad():
    for batch in tqdm(dataloader, desc="Computing ESM embeddings"):

        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True).to(device)


        outputs = model(**inputs)

        last_hidden_state = outputs.last_hidden_state
        attention_mask = inputs['attention_mask']
        mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        sum_embeddings = torch.sum(last_hidden_state * mask_expanded, 1)
        sum_mask = torch.clamp(mask_expanded.sum(1), min=1e-9)
        mean_embeddings = (sum_embeddings / sum_mask).cpu().numpy()

        x.append(mean_embeddings)

# combine them into the final matrix
x = np.vstack(x)
#y = hl_df['Half-life, h'].values

print(f"Ready! The resulting matrix is: {x.shape}")
x_df = pd.DataFrame(x)
x_df['Sequence'] = hl_df['Sequence']
x_df['y'] = hl_df[func_y]
x_df.to_csv(f'{path}/{folder}/{data_name}_{set_name}_esm2_{num_model}.csv', index=None)